# 🌍 Cross-Country Climate Vulnerability Analysis (NASA POWER)
This notebook compares Ethiopia, Kenya, and Nigeria to compute a climate vulnerability ranking for COP32 analysis.

## Temperature Trend Comparison

The monthly temperature trends show seasonal patterns across countries. 
Some countries experience consistently higher temperatures, indicating greater heat exposure.

The variation over time suggests differences in climate stability and warming trends.

## Precipitation Variability

The boxplots reveal differences in rainfall distribution across countries.

Countries with wider spreads and outliers show higher variability, 
indicating less predictable rainfall patterns and higher climate risk.

## Time Series Analysis

Temperature shows a clear seasonal pattern, with identifiable warm and cool periods.

Rainfall is more irregular, with sharp peaks representing rainy seasons.

These patterns highlight climate variability and seasonal dependence.

## Correlation Analysis

The heatmap shows relationships between climate variables.

Strong correlations indicate linked climate behaviors, 
while weak correlations suggest independent factors.

Understanding these relationships helps explain climate dynamics and risks.

## Climate Vulnerability Ranking

The vulnerability index combines temperature, rainfall variability, 
temperature range, and wind speed.

Countries with higher scores are more exposed to climate instability.

This ranking supports climate policy decisions and COP32 discussions.

## Final Conclusion

This cross-country analysis highlights significant differences in climate behavior.

Temperature trends, rainfall variability, and correlations all point to varying levels of climate risk.

The vulnerability ranking provides a data-driven basis for understanding climate exposure 
and supports informed decision-making for COP32.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

# ==============================
# 1. LOAD DATA
# ==============================
ethiopia = pd.read_csv("../data/ethiopia_clean.csv")
kenya = pd.read_csv("../data/kenya_clean.csv")
nigeria = pd.read_csv("../data/nigeria_clean.csv")

ethiopia['country'] = 'Ethiopia'
kenya['country'] = 'Kenya'
nigeria['country'] = 'Nigeria'

df = pd.concat([ethiopia, kenya, nigeria], ignore_index=True)
df.columns = df.columns.str.lower().str.strip()

# ==============================
# 2. CREATE PROPER DATE
# ==============================
df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
df = df.sort_values('date')

# ==============================
# 3. MONTHLY TEMPERATURE
# ==============================
monthly = df.groupby(['date', 'country'])['t2m'].mean().reset_index()

plt.figure(figsize=(12,6))

for c in monthly['country'].unique():
    subset = monthly[monthly['country'] == c]
    plt.plot(subset['date'], subset['t2m'], label=c)

plt.title("Monthly Average Temperature Comparison (2015–2026)")
plt.xlabel("Date")
plt.ylabel("Temperature (T2M)")
plt.legend()
plt.show()

# ==============================
# 4. FEATURE ENGINEERING
# ==============================
df['t2m_range'] = df['t2m_max'] - df['t2m_min']

# ==============================
# 5. COUNTRY SUMMARY
# ==============================
summary = df.groupby('country').agg({
    't2m': ['mean', 'median', 'std']
})

summary.columns = ['mean_temp', 'median_temp', 'std_temp']
summary = summary.reset_index()

print("\nTemperature Summary:")
print(summary)

# ==============================
# 6. VULNERABILITY INDEX
# ==============================
vuln = df.groupby('country').agg({
    't2m': 'mean',
    'prectotcorr': 'std',
    't2m_range': 'mean',
    'ws2m': 'mean'
})

vuln.columns = ['avg_temp', 'rain_var', 'temp_range', 'wind']
vuln = vuln.reset_index()

scaler = MinMaxScaler()
features = ['avg_temp', 'rain_var', 'temp_range', 'wind']
vuln[features] = scaler.fit_transform(vuln[features])

vuln['vulnerability'] = (
    vuln['avg_temp'] * 0.35 +
    vuln['rain_var'] * 0.30 +
    vuln['temp_range'] * 0.20 +
    vuln['wind'] * 0.15
)

ranking = vuln.sort_values('vulnerability', ascending=False)

print("\nVulnerability Ranking:")
print(ranking[['country', 'vulnerability']])

# ==============================
# 7. PLOT RANKING
# ==============================
ranking.set_index('country')['vulnerability'].plot(kind='bar')
plt.title("Climate Vulnerability Ranking")
plt.show()
# ==============================
# 8. PRECIPITATION VARIABILITY
# ==============================
import seaborn as sns

plt.figure(figsize=(10,6))
sns.boxplot(data=df, x='country', y='prectotcorr')

plt.title("Precipitation Variability Across Countries")
plt.xlabel("Country")
plt.ylabel("PRECTOTCORR")
plt.show()

# Summary stats
rain_summary = df.groupby('country')['prectotcorr'].agg(['mean', 'median', 'std']).reset_index()
rain_summary.columns = ['country', 'mean_rain', 'median_rain', 'std_rain']

print("\nPrecipitation Summary:")
print(rain_summary)
# ==============================
# 9. TIME SERIES ANALYSIS
# ==============================

# Monthly average temperature (ALL countries combined)
monthly_temp = df.groupby('date')['t2m'].mean()

plt.figure(figsize=(12,5))
monthly_temp.plot()

# Find extremes
max_temp = monthly_temp.max()
min_temp = monthly_temp.min()

max_date = monthly_temp.idxmax()
min_date = monthly_temp.idxmin()

# Annotate
plt.scatter(max_date, max_temp)
plt.text(max_date, max_temp, 'Warmest')

plt.scatter(min_date, min_temp)
plt.text(min_date, min_temp, 'Coolest')

plt.title("Monthly Average Temperature")
plt.show()


# Monthly rainfall
monthly_rain = df.groupby('date')['prectotcorr'].sum()

plt.figure(figsize=(12,5))
monthly_rain.plot(kind='bar')

peak_date = monthly_rain.idxmax()
peak_value = monthly_rain.max()

plt.text(list(monthly_rain.index).index(peak_date), peak_value, 'Peak Rain', rotation=90)

plt.title("Monthly Total Rainfall")
plt.show()
# ==============================
# 10. CORRELATION ANALYSIS
# ==============================

numeric_df = df.select_dtypes(include='number')

plt.figure(figsize=(10,6))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm')

plt.title("Correlation Heatmap")
plt.show()
# ==============================
# 11. SCATTER PLOTS
# ==============================

plt.figure()
plt.scatter(df['t2m'], df['rh2m'])
plt.xlabel("T2M")
plt.ylabel("RH2M")
plt.title("Temperature vs Humidity")
plt.show()

plt.figure()
plt.scatter(df['t2m_range'], df['ws2m'])
plt.xlabel("T2M Range")
plt.ylabel("WS2M")
plt.title("Temperature Range vs Wind Speed")
plt.show()
# ==============================
# 12. TOP 3 CORRELATIONS
# ==============================
corr_matrix = numeric_df.corr().abs()

# Remove self-correlation
np.fill_diagonal(corr_matrix.values, 0)

top_corr = corr_matrix.unstack().sort_values(ascending=False).drop_duplicates()

print("\nTop 3 Strongest Correlations:")
print(top_corr.head(3))